# BeeSpace 00 - Pipeline integrado

Este notebook demonstra a jornada completa da BeeSpace: colmeia georreferenciada, raio potencial de biovigilância de 3 km, dados Copernicus simulados, sinais locais da colmeia e classificação de risco.

Os dados são sintéticos. Em uma versão operacional, serão substituídos por dados do Copernicus Data Space Ecosystem, sensores IoT, visão computacional, bioacústica e registros de campo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
RAIO_VOO_KM = 3
AREA_KM2 = np.pi * RAIO_VOO_KM**2
AREA_HECTARES = AREA_KM2 * 100

print(f"Raio potencial de análise: {RAIO_VOO_KM} km")
print(f"Área potencial de biovigilância: {AREA_KM2:.2f} km²")
print(f"Área potencial de biovigilância: {AREA_HECTARES:.0f} hectares")

In [ ]:
colmeias = pd.DataFrame({
    "id_colmeia": ["C001", "C002", "C003", "C004", "C005"],
    "latitude": [-28.9368, -28.9482, -28.9275, -28.9601, -28.9184],
    "longitude": [-51.5489, -51.5620, -51.5303, -51.5755, -51.5208],
    "produtor": ["Apiário A", "Apiário B", "Apiário C", "Apiário D", "Apiário E"]
})
colmeias["raio_km"] = RAIO_VOO_KM
colmeias["area_potencial_ha"] = round(AREA_HECTARES, 0)
colmeias

## Dados Copernicus simulados

As variáveis representam Sentinel-2, CLMS Land Cover, C3S/ERA5-Land e Sentinel-5P/CAMS.

In [ ]:
n = len(colmeias)

dados_copernicus = pd.DataFrame({
    "id_colmeia": colmeias["id_colmeia"],
    "ndvi": np.random.uniform(0.20, 0.85, n),
    "evi": np.random.uniform(0.15, 0.75, n),
    "ndwi": np.random.uniform(-0.20, 0.50, n),
    "temp_media": np.random.uniform(18, 36, n),
    "precipitacao_7d": np.random.uniform(0, 90, n),
    "umidade_solo": np.random.uniform(0.10, 0.65, n),
    "poluicao_indice": np.random.uniform(0, 1, n),
    "perc_mata_nativa": np.random.uniform(0.05, 0.80, n),
    "perc_agricultura": np.random.uniform(0.05, 0.80, n),
    "perc_solo_exposto": np.random.uniform(0.01, 0.30, n)
})

soma = dados_copernicus[["perc_mata_nativa", "perc_agricultura", "perc_solo_exposto"]].sum(axis=1)
for col in ["perc_mata_nativa", "perc_agricultura", "perc_solo_exposto"]:
    dados_copernicus[col] = dados_copernicus[col] / soma

dados_copernicus

In [ ]:
dados_colmeia = pd.DataFrame({
    "id_colmeia": colmeias["id_colmeia"],
    "temp_colmeia": np.random.uniform(28, 40, n),
    "umidade_colmeia": np.random.uniform(38, 80, n),
    "variacao_peso_7d": np.random.uniform(-3.5, 4.5, n),
    "atividade_abelhas": np.random.uniform(0.10, 1.00, n),
    "anomalia_acustica": np.random.uniform(0.00, 1.00, n),
    "mortalidade_observada": np.random.uniform(0.00, 1.00, n)
})
dados_colmeia

In [ ]:
base = colmeias.merge(dados_copernicus, on="id_colmeia").merge(dados_colmeia, on="id_colmeia")
base

## Classificação de risco

A pontuação abaixo simula a lógica de alerta. Em operação, esta etapa pode usar o modelo de Machine Learning treinado com dados reais.

In [ ]:
score = (
    (base["ndvi"] < 0.35).astype(int) * 2
    + (base["ndwi"] < 0.00).astype(int)
    + (base["precipitacao_7d"] < 15).astype(int)
    + (base["umidade_solo"] < 0.20).astype(int)
    + (base["temp_media"] > 33).astype(int)
    + (base["temp_colmeia"] > 37).astype(int) * 2
    + (base["variacao_peso_7d"] < -1.0).astype(int) * 2
    + (base["atividade_abelhas"] < 0.35).astype(int) * 2
    + (base["anomalia_acustica"] > 0.65).astype(int) * 2
    + (base["mortalidade_observada"] > 0.55).astype(int) * 3
    + (base["poluicao_indice"] > 0.70).astype(int)
    + (base["perc_agricultura"] > 0.60).astype(int)
    + (base["perc_solo_exposto"] > 0.25).astype(int)
    + (base["perc_mata_nativa"] < 0.20).astype(int)
)
base["score_risco"] = score
base["status_beespace"] = pd.cut(score, bins=[-1, 3, 7, 99], labels=["normal", "atencao", "alerta"])
base[["id_colmeia", "produtor", "score_risco", "status_beespace"]]

In [ ]:
def gerar_interpretacao(row):
    fatores = []
    if row["ndvi"] < 0.35:
        fatores.append("baixo vigor vegetal")
    if row["precipitacao_7d"] < 15:
        fatores.append("baixa precipitação")
    if row["variacao_peso_7d"] < -1.0:
        fatores.append("queda de peso")
    if row["atividade_abelhas"] < 0.35:
        fatores.append("baixa atividade")
    if row["anomalia_acustica"] > 0.65:
        fatores.append("anomalia acústica")
    if row["mortalidade_observada"] > 0.55:
        fatores.append("mortalidade observada")
    return ", ".join(fatores) if fatores else "sem fatores críticos relevantes"

base["fatores_alerta"] = base.apply(gerar_interpretacao, axis=1)
base[["id_colmeia", "status_beespace", "fatores_alerta"]]

In [ ]:
contagem = base["status_beespace"].value_counts().reindex(["normal", "atencao", "alerta"])
plt.figure(figsize=(7, 4))
plt.bar(contagem.index.astype(str), contagem.values)
plt.title("Status BeeSpace por colmeia")
plt.xlabel("Status")
plt.ylabel("Quantidade")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "status_beespace_colmeias.png", dpi=150)
plt.show()

base.to_csv(OUTPUT_DIR / "resultado_pipeline_integrado_beespace.csv", index=False)

## Síntese

O satélite observa o território, a colmeia sente o ambiente e a inteligência de dados transforma sinais em decisão.